In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kaushalnandania/student-placement-dataset/0000.parquet


In [3]:

import kagglehub
# Download latest version
path = kagglehub.dataset_download("kaushalnandania/student-placement-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/kaushalnandania/student-placement-dataset


In [4]:
import os

print(os.listdir(path))

['0000.parquet']


In [5]:
import pandas as pd

df = pd.read_parquet("/kaggle/input/datasets/kaushalnandania/student-placement-dataset/0000.parquet")

In [6]:
df.head()

,Internships,CGPA,HistoryOfBacklogs,PlacedOrNot
0,1,8,1,1
1,0,7,1,1
2,1,6,0,1
3,0,8,1,1
4,0,8,0,1


In [7]:
X=df.drop(columns=['PlacedOrNot']).values
Y=df['PlacedOrNot'].values

In [8]:
X

array([[1, 8, 1],
       [0, 7, 1],
       [1, 6, 0],
       ...,
       [1, 7, 0],
       [1, 7, 0],
       [0, 8, 0]], shape=(2966, 3))

In [9]:
Y

array([1, 1, 1, ..., 0, 0, 1], shape=(2966,))

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim 
from sklearn.model_selection import train_test_split
from torch .utils.data import DataLoader,TensorDataset


In [11]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2)

In [12]:
X_train

array([[0, 7, 1],
       [1, 7, 0],
       [1, 7, 0],
       ...,
       [0, 6, 0],
       [0, 7, 0],
       [0, 8, 0]], shape=(2372, 3))

In [13]:
Y_train

array([0, 1, 0, ..., 1, 0, 1], shape=(2372,))

In [14]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()

X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled= scaler.transform(X_test)


In [15]:
X_train_scaled

array([[-0.94669038, -0.08272077,  2.0809462 ],
       [ 0.40686573, -0.08272077, -0.48055063],
       [ 0.40686573, -0.08272077, -0.48055063],
       ...,
       [-0.94669038, -1.11001745, -0.48055063],
       [-0.94669038, -0.08272077, -0.48055063],
       [-0.94669038,  0.94457591, -0.48055063]], shape=(2372, 3))

In [16]:
X_train_scaled_tensor=torch.tensor(X_train_scaled,dtype=torch.float32)
X_test_scaled_tensor=torch.tensor(X_test_scaled,dtype=torch.float32)
Y_train_tensor=torch.tensor(Y_train,dtype=torch.float32).unsqueeze(1)
Y_test_tensor=torch.tensor(Y_test,dtype=torch.float32).unsqueeze(1)


In [17]:
train_dataset=TensorDataset(X_train_scaled_tensor,Y_train_tensor)

In [18]:
X_train_scaled_tensor.shape

torch.Size([2372, 3])

In [19]:
train_loader=DataLoader(train_dataset,batch_size=50,shuffle=True)

In [64]:
class placement(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(3,10)
        self.fc2=nn.Linear(10,6)
        self.fc3=nn.Linear(6,5)
        self.fc4=nn.Linear(5,1)

        self.relu=nn.ReLU()
        self.sigmoid=nn.Sigmoid()

    def forward(self,x):
        x=self.relu(self.fc1(x))
        x=self.relu(self.fc2(x))
        x=self.relu(self.fc3(x))
        x=self.sigmoid(self.fc4(x))

        return x
        



In [65]:
model=placement()
print(model)

placement(
  (fc1): Linear(in_features=3, out_features=10, bias=True)
  (fc2): Linear(in_features=10, out_features=6, bias=True)
  (fc3): Linear(in_features=6, out_features=5, bias=True)
  (fc4): Linear(in_features=5, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)


In [66]:
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.1)

In [67]:
epochs=100
for epoch in range(epochs):

    for x_batch ,y_batch in train_loader:

        y_pred=model(x_batch)

        loss=criterion(y_pred,y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch[ {epoch+1}/{epochs}],Loss:{loss.item():4f}")
    

Epoch[ 1/100],Loss:0.287127
Epoch[ 2/100],Loss:0.443793
Epoch[ 3/100],Loss:0.371216
Epoch[ 4/100],Loss:0.366954
Epoch[ 5/100],Loss:0.368153
Epoch[ 6/100],Loss:0.420115
Epoch[ 7/100],Loss:0.396576
Epoch[ 8/100],Loss:0.307891
Epoch[ 9/100],Loss:0.321133
Epoch[ 10/100],Loss:0.407085
Epoch[ 11/100],Loss:0.394466
Epoch[ 12/100],Loss:0.461637
Epoch[ 13/100],Loss:0.618699
Epoch[ 14/100],Loss:0.386591
Epoch[ 15/100],Loss:0.441305
Epoch[ 16/100],Loss:0.253760
Epoch[ 17/100],Loss:0.493925
Epoch[ 18/100],Loss:0.526602
Epoch[ 19/100],Loss:0.503999
Epoch[ 20/100],Loss:0.671236
Epoch[ 21/100],Loss:0.297014
Epoch[ 22/100],Loss:0.335659
Epoch[ 23/100],Loss:0.268247
Epoch[ 24/100],Loss:0.486656
Epoch[ 25/100],Loss:0.505162
Epoch[ 26/100],Loss:0.324138
Epoch[ 27/100],Loss:0.434921
Epoch[ 28/100],Loss:0.425946
Epoch[ 29/100],Loss:0.256185
Epoch[ 30/100],Loss:0.346735
Epoch[ 31/100],Loss:0.373087
Epoch[ 32/100],Loss:0.296881
Epoch[ 33/100],Loss:0.421992
Epoch[ 34/100],Loss:0.458674
Epoch[ 35/100],Loss:0.3

In [68]:
with torch.no_grad():
    correct=0
    total=0
    model.eval()
    preds=model(X_test_scaled_tensor)
    loss=criterion(preds,Y_test_tensor)
    predicted=(preds>=0.5).float()
    accuracy=(predicted==Y_test_tensor).float().mean()
print(f"ACCURACY:{accuracy.item()*100:.2f}%")

ACCURACY:83.84%
